# Sharing Graphical Parameters

This tutorial demonstrates how to link graphical objects across panels
by giving them the same name, enabling simultaneous editing. It follows
the R vignette *An Example of "Linking" Graphical Objects in grid*.

Key idea: when multiple grobs on the display list share the same name,
`grid_edit(..., global_=True)` edits all of them at once. This is
useful for shared axes across multipanel plots.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import grid_py as gp

## Two panels with shared axis names

We draw two side-by-side scatter plots.  Both x-axes are named
`"xaxis"` and both y-axes are named `"yaxis"`, so they can be edited
simultaneously.

In [ ]:
rng = np.random.default_rng(0)
x = np.arange(1, 11)
y1 = rng.standard_normal(10)
y2 = rng.standard_normal(10)

gp.grid_newpage()

# Outer layout: 1 row, 2 columns
gp.push_viewport(
    gp.Viewport(layout=gp.GridLayout(nrow=1, ncol=2, respect=True))
)

# --- Panel 1 ---
vp1a = gp.Viewport(layout_pos_col=1)
vp1b = gp.Viewport(
    width=0.6, height=0.6,
    xscale=[0, 11], yscale=[-4, 4],
)
gp.push_viewport(vp1a)
gp.push_viewport(vp1b)
gp.grid_xaxis(name="xaxis")
gp.grid_yaxis(name="yaxis")
gp.grid_points(
    gp.Unit(x.tolist(), "native"),
    gp.Unit(y1.tolist(), "native"),
)
gp.pop_viewport(2)

# --- Panel 2 ---
vp2a = gp.Viewport(layout_pos_col=2)
vp2b = gp.Viewport(
    width=0.6, height=0.6,
    xscale=[0, 11], yscale=[-4, 4],
)
gp.push_viewport(vp2a)
gp.push_viewport(vp2b)
gp.grid_xaxis(name="xaxis")
gp.grid_yaxis(name="yaxis")
gp.grid_points(
    gp.Unit(x.tolist(), "native"),
    gp.Unit(y2.tolist(), "native"),
)
gp.pop_viewport(2)

plt.gcf()

Both panels now have identical axes with the default tick positions.

## Simultaneous axis editing with `grid_edit`

Because both x-axes share the name `"xaxis"`, a single `grid_edit`
call with `global_=True` updates them both at once.

In [ ]:
gp.grid_edit("xaxis", at=[1, 5, 9], global_=True)
plt.gcf()

Both panels now show tick marks only at 1, 5, and 9 instead of the
original default positions -- achieved with a single edit call.

## Off-screen approach using edit_grob

For an entirely off-screen workflow, you can build grob objects
and use `edit_grob` to modify axes before drawing.

In [ ]:
# Build axis grobs off-screen
xa = gp.xaxis_grob(name="xaxis")
ya = gp.yaxis_grob(name="yaxis")

# Edit tick positions
xa = gp.edit_grob(xa, at=[1, 5, 9])

# Edit colour of both axes
xa = gp.edit_grob(xa, gp=gp.Gpar(col="darkblue"))
ya = gp.edit_grob(ya, gp=gp.Gpar(col="darkblue"))

# Draw in a viewport
gp.grid_newpage()
gp.push_viewport(gp.Viewport(
    width=0.5, height=0.5,
    xscale=[0, 11], yscale=[-4, 4],
))
gp.grid_draw(xa)
gp.grid_draw(ya)
gp.grid_points(
    gp.Unit(x.tolist(), "native"),
    gp.Unit(y1.tolist(), "native"),
    gp=gp.Gpar(col="steelblue"),
)
gp.pop_viewport()

plt.gcf()

This off-screen approach is especially useful for building reusable
plotting functions where shared graphical parameters ensure visual
consistency across panels -- for example, in a scatterplot matrix.